In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

In [ ]:
df = pd.read_csv('wheat_prices.csv')
time = np.arange(len(df))
prices = df['Price'].values

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices.reshape(-1,
1))
# Create sequences
def create_sequences(data, seq_length):
 X, y = [], []
 for i in range(len(data) - seq_length):
 X.append(data[i:i+seq_length])
 y.append(data[i+seq_length])
 return np.array(X), np.array(y)
seq_length = 60
X, y = create_sequences(prices_scaled, seq_length)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

In [ ]:
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [ ]:
model = Sequential([
 LSTM(50, activation='relu',
return_sequences=False, input_shape=(seq_length, 1)),
 Dense(1)
])
model.compile(optimizer='adam', loss='mse')

In [ ]:
model.fit(X_train, y_train, epochs=50, batch_size=32,
validation_data=(X_test, y_test))

In [ ]:
y_pred = model.predict(X_test)

y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time[seq_length+train_size:], y_test_inv,
label='Actual Prices', color='blue')
plt.plot(time[seq_length+train_size:], y_pred_inv,
label='Predicted Prices', color='red', linestyle='dashed')
plt.xlabel('Month')
plt.ylabel('Wheat Price/ton')
plt.legend()
plt.title(' Wheat price prediction using Lstm')
plt.show()